In [11]:
import openai
import pinecone
import json

from pinecone import Pinecone
from openai import OpenAI

In [12]:
import os

from dotenv import load_dotenv
load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key = pinecone_api_key)
print("Pinecone client successfully configured.")
print(pinecone_api_key[:5])

openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key = openai_api_key)
print("OpenAI client successfully configured.")
print(openai_api_key[:5])

index_name = "faq-database"
index = pc.Index(index_name)

Pinecone client successfully configured.
pcsk_
OpenAI client successfully configured.
sk-pr


In [13]:
def embedding_model(query, openai_client, model="text-embedding-3-small"):
  response = openai_client.embeddings.create(
      model=model,
      input=query
  )

  embedding = response.data[0].embedding
  return embedding

In [14]:
system_prompt = {
                    "role": "system",
                    "content": f"""
                    You are a helpfull E-Commerce assistant helping customers with their general questions regarding policies and procedures when buying in our store.
                    Our store sells e-books and courses for IT professionals.
                    """,
                }

def prompt_builder(system_message, context):
  return system_message["content"].format(context)

In [15]:
def candidates_generation(query, openai_client, n_candidates=2):

  system_prompt = f"""
You are an AI assistant that generates {n_candidates} alternative phrasings of the user's question.
These alternatives will be used to retrieve relevant documents from a vector database.
Keep each question short and to the point.

Respond ONLY with valid JSON in this exact format, with exactly {n_candidates} entries:
{{
    "candidates": ["<alternative question 1>", "<alternative question 2>"]
}}

Original question:
{query}
"""

  messages = [{"role": "system", "content": system_prompt}]

  response = openai_client.chat.completions.create(
      model="gpt-4o",
      messages=messages,
      max_tokens=1500,
      response_format={ "type": "json_object" }
    )

  response_content = response.choices[0].message.content
  response_type = json.loads(response_content)
  return response_type["candidates"]

In [16]:
def retrieve_faq(query_embedding, index, top_k=1):
    response = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        namespace="ns1"
    )
    return response['matches'][0]['metadata']['answer']

In [17]:
def combine_documents(retrieved_docs):
    return "\n\n".join(retrieved_docs)

In [18]:
def multi_query_rag_chatbot(query, openai_client, index):

    candidates = candidates_generation(query, openai_client, n_candidates=2)

    relevant_docs = []
    for candidate in candidates:
      candidate_embedding = embedding_model(candidate, openai_client)
      best_match = retrieve_faq(candidate_embedding, index)
      relevant_docs.append(best_match)

    context = combine_documents(relevant_docs)

    augmented_prompt = prompt_builder(system_prompt, context)

    messages = [{"role": "system","content": augmented_prompt},
                {"role": "user","content": query}]

    response = openai_client.chat.completions.create(
      model="gpt-4o",
      messages=messages,
      max_tokens=250
    )

    return response.choices[0].message.content

In [19]:
query = "do you offer any delivery"

response = multi_query_rag_chatbot(query,client, index)

print(f"User: {query}")
print(f"Bot: {response}")

User: do you offer any delivery
Bot: As our store specializes in e-books and courses for IT professionals, all of our products are digital. This means there is no physical delivery involved. Once you complete your purchase, you will have instant access to download your e-books or access your courses online. If you encounter any issues accessing your digital products, feel free to reach out for assistance!
